 SHL Grammar Scoring

This notebook shifts from a simple text pipeline to an evidence-driven research framework, evaluating Audio Embeddings, Acoustic Features, and Advanced Text Features independently before ensembling.

In [ ]:
# 1. INSTALL DEPENDENCIES
!pip install -q pandas numpy scikit-learn librosa soundfile openai-whisper spacy sentence-transformers catboost xgboost lightgbm optuna transformers
!python -m spacy download en_core_web_sm
!apt-get update && apt-get install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 13.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 104.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelea

In [ ]:
# 2. CONFIGURATION & MOUNT
import os
import warnings
import pandas as pd
import numpy as np
import torch
import librosa
import whisper
import spacy
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error
from scipy.stats import pearsonr
import lightgbm as lgb
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

DATASET_DIR = "/content/drive/MyDrive/Dataset_Final"
os.makedirs("cache", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

df_train = pd.read_csv(f"{DATASET_DIR}/train.csv")
df_test = pd.read_csv(f"{DATASET_DIR}/test.csv")
df_train['audio_path'] = df_train['filename'].apply(lambda x: f"{DATASET_DIR}/train/{x}")
df_test['audio_path'] = df_test['filename'].apply(lambda x: f"{DATASET_DIR}/test/{x}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dataset Loaded. Using device: {device}")

Mounted at /content/drive
Dataset Loaded. Using device: cuda


## Phase 1: Acoustic Features (MFCC, Silence, Pitch)

In [ ]:
def extract_acoustic(df, cache_file):
    if os.path.exists(cache_file): return np.load(cache_file)

    features = []
    for path in tqdm(df['audio_path'], desc="Acoustic"):
        try:
            y, sr = librosa.load(path, sr=16000)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13).mean(axis=1)
            zcr = librosa.feature.zero_crossing_rate(y).mean()
            # simple silence ratio (energy below threshold)
            rms = librosa.feature.rms(y=y)[0]
            silence_ratio = np.mean(rms < 0.01)
            feat = np.concatenate([mfcc, [zcr, silence_ratio]])
            features.append(feat)
        except:
            features.append(np.zeros(15))

    feats = np.array(features)
    np.save(cache_file, feats)
    return feats

print("Extracting Acoustic Features...")
X_train_ac = extract_acoustic(df_train, "cache/train_acoustic.npy")
X_test_ac = extract_acoustic(df_test, "cache/test_acoustic.npy")

Extracting Acoustic Features...


Acoustic:   0%|          | 0/769 [00:00<?, ?it/s]

Acoustic:   0%|          | 0/216 [00:00<?, ?it/s]

## Phase 2: Speech Embeddings (Wav2Vec2)

In [ ]:
def extract_speech_embeddings(df, cache_file):
    if os.path.exists(cache_file): return np.load(cache_file)

    processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
    model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device)

    features = []
    for path in tqdm(df['audio_path'], desc="Wav2Vec2"):
        try:
            y, sr = librosa.load(path, sr=16000)
            inputs = processor(y, sampling_rate=16000, return_tensors="pt").to(device)
            with torch.no_grad():
                out = model(**inputs).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
            features.append(out)
        except:
            features.append(np.zeros(768))

    feats = np.array(features)
    np.save(cache_file, feats)
    return feats

print("Extracting Speech Embeddings...")
X_train_sp = extract_speech_embeddings(df_train, "cache/train_speech.npy")
X_test_sp = extract_speech_embeddings(df_test, "cache/test_speech.npy")

Extracting Speech Embeddings...


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Wav2Vec2:   0%|          | 0/769 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Wav2Vec2:   0%|          | 0/216 [00:00<?, ?it/s]

## Phase 3: Text Transcripts & Grammar (Whisper + Spacy)

In [ ]:
def extract_text_features(df, cache_prefix):
    txt_cache = f"cache/{cache_prefix}_transcripts.csv"
    emb_cache = f"cache/{cache_prefix}_text_emb.npy"
    gram_cache = f"cache/{cache_prefix}_grammar.npy"

    if os.path.exists(emb_cache) and os.path.exists(gram_cache):
        return np.load(gram_cache), np.load(emb_cache)

    # Transcribe (Using base instead of tiny for better accuracy)
    asr_model = whisper.load_model("base").to(device)
    transcripts = []
    for path in tqdm(df['audio_path'], desc="Whisper Base"):
        try:
            transcripts.append(asr_model.transcribe(path, fp16=(device=="cuda"))["text"].strip())
        except:
            transcripts.append("")
    df['transcript'] = transcripts
    df[['filename', 'transcript']].to_csv(txt_cache, index=False)

    # Text Embeddings
    emb_model = SentenceTransformer("all-mpnet-base-v2")
    emb = emb_model.encode(transcripts, show_progress_bar=True)
    np.save(emb_cache, emb)

    # Grammar / POS
    nlp = spacy.load("en_core_web_sm")
    grammar = []
    for t in tqdm(transcripts, desc="Grammar"):
        doc = nlp(t)
        words = [tok for tok in doc if not tok.is_punct]
        pos_counts = {pos: 0 for pos in ['NOUN', 'VERB', 'ADJ', 'ADV', 'PRON']}
        for tok in doc:
            if tok.pos_ in pos_counts: pos_counts[tok.pos_] += 1

        wc = len(words)
        grammar.append([
            wc, len(list(doc.sents)),
            pos_counts['NOUN']/max(wc,1), pos_counts['VERB']/max(wc,1),
            pos_counts['ADJ']/max(wc,1), pos_counts['PRON']/max(wc,1)
        ])

    gram = np.array(grammar)
    np.save(gram_cache, gram)
    return gram, emb

print("Extracting Text & Grammar...")
X_train_gr, X_train_te = extract_text_features(df_train, "train")
X_test_gr, X_test_te = extract_text_features(df_test, "test")

Extracting Text & Grammar...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 191MiB/s]


Whisper Base:   0%|          | 0/769 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

Grammar:   0%|          | 0/769 [00:00<?, ?it/s]

Whisper Base:   0%|          | 0/216 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Grammar:   0%|          | 0/216 [00:00<?, ?it/s]

## Phase 4: Feature Selection & PCA
Feeding 768+ dimensions directly to trees on 769 rows causes overfitting. We use PCA on embeddings.

In [ ]:
# PCA on Speech Embeddings
pca_sp = PCA(n_components=30, random_state=42)
X_train_sp_pca = pca_sp.fit_transform(StandardScaler().fit_transform(X_train_sp))
X_test_sp_pca = pca_sp.transform(StandardScaler().fit_transform(X_test_sp))

# PCA on Text Embeddings
pca_te = PCA(n_components=30, random_state=42)
X_train_te_pca = pca_te.fit_transform(StandardScaler().fit_transform(X_train_te))
X_test_te_pca = pca_te.transform(StandardScaler().fit_transform(X_test_te))

# Combined Features for Trees
X_train_all = np.hstack([X_train_ac, X_train_gr, X_train_sp_pca, X_train_te_pca])
X_test_all = np.hstack([X_test_ac, X_test_gr, X_test_sp_pca, X_test_te_pca])
y_train = df_train['label'].values
print("Combined Feature Shape:", X_train_all.shape)

Combined Feature Shape: (769, 81)


## Phase 5: Model Benchmarking Framework

In [ ]:
def evaluate_model(X, y, model, name):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmses, pearsons = [], []
    oof = np.zeros(len(y))

    for tr_idx, va_idx in kf.split(X):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        # Scaling inside fold
        scaler = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_tr)
        X_va_sc = scaler.transform(X_va)

        model.fit(X_tr_sc, y_tr)
        preds = np.clip(model.predict(X_va_sc), 0, 5)
        oof[va_idx] = preds
        rmses.append(root_mean_squared_error(y_va, preds))
        pearsons.append(0.0 if np.std(preds)==0 else pearsonr(y_va, preds)[0])

    return {"Model": name, "Mean CV RMSE": np.mean(rmses), "OOF RMSE": root_mean_squared_error(y, oof), "OOF Pearson": pearsonr(y, oof)[0], "OOF Preds": oof}

models = [
    (Ridge(alpha=10.0), "Ridge"),
    (SVR(C=1.0, kernel='rbf'), "SVR"),
    (CatBoostRegressor(verbose=False, random_state=42), "CatBoost"),
    (lgb.LGBMRegressor(verbosity=-1, random_state=42), "LightGBM")
]

results = []
for m, name in models:
    res = evaluate_model(X_train_all, y_train, m, name)
    results.append(res)

df_res = pd.DataFrame(results).drop(columns=["OOF Preds"]).sort_values("OOF RMSE")
display(df_res)

,Model,Mean CV RMSE,OOF RMSE,OOF Pearson
2,CatBoost,0.662769,0.663239,0.849632
0,Ridge,0.670256,0.670584,0.840759
3,LightGBM,0.677206,0.678059,0.837185
1,SVR,0.712131,0.712475,0.824075


## Phase 6: Weighted Ensemble Prediction

In [ ]:
# Simple average of top 2 models (e.g. SVR and CatBoost)
# In a full Grandmaster run, weights would be optimized via Optuna on OOF predictions.
best_model_1 = SVR(C=1.0, kernel='rbf')
best_model_2 = CatBoostRegressor(verbose=False, random_state=42)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train_all)
X_te_sc = scaler.transform(X_test_all)

best_model_1.fit(X_tr_sc, y_train)
best_model_2.fit(X_tr_sc, y_train)

p1 = np.clip(best_model_1.predict(X_te_sc), 0, 5)
p2 = np.clip(best_model_2.predict(X_te_sc), 0, 5)

final_preds = (p1 * 0.5) + (p2 * 0.5)

sub = df_test[['filename']].copy()
sub['label'] = final_preds
sub.to_csv("outputs/GRANDMASTER_SUBMISSION.csv", index=False)
print("Saved outputs/GRANDMASTER_SUBMISSION.csv")
display(sub.head())

Saved outputs/GRANDMASTER_SUBMISSION.csv


,filename,label
0,audio_128.wav,3.668589
1,audio_156.wav,3.748071
2,audio_19.wav,4.274845
3,audio_108.wav,2.462770
4,audio_206.wav,2.456175
